## Import packages 

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

In [ ]:
# Implement this code in order to make any figures always use Times New Roman. Otherwise when you do it in the figures the matplotlib default can override it
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12

# Set figure titles to be the same
mpl.rcParams['axes.titlesize'] = 20
mpl.rcParams['axes.titleweight'] = 'bold'
mpl.rcParams['axes.titlepad'] = 20
mpl.rcParams['font.family'] = 'Times New Roman'

## Set up base paths and paths to read in files

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
land_use = base_path / "Inputs/2013_landuse_LandCover.shp"
terrestrial_landcover = gpd.read_file(land_use)

In [ ]:
hydrobasins = base_path / "Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp"
hydrobasins = gpd.read_file(hydrobasins)
print(hydrobasins.crs)

## Set up output folder 

In [ ]:
# Define the output folder path
output_folder = base_path / "Processed_data/existing_future_forest"

In [ ]:
map_forest_connectivity_figures_folder = base_path / "Results/Map_Figures/forest_connectivity"

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
jamaica_boundary_path = base_path / "Inputs/Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(f"Original Jamaica boundary CRS: {jamaica_boundary.crs}")

In [ ]:
# Reproject to Jamaica Metric Grid (EPSG:3448)
terrestrial_landcover = terrestrial_landcover.to_crs(jamaica_metric_grid_crs)
print("Landcover CRS:", terrestrial_landcover.crs)

## Define which classes count as "Afforestable non-agricultural lands"

In [ ]:
afforestable_non_agri_non_other_ecosystem_classes = [
    'Fields: Bare Land',
    'Quarry',
    'Bauxite Extraction',
]

## Filter afforestable non agri non other ecosystem classes

In [ ]:
afforestable_non_agri_non_other_ecosystem_classes = terrestrial_landcover[terrestrial_landcover["Classify"].isin(afforestable_non_agri_non_other_ecosystem_classes)].copy()
print(f"Number of forest polygons: {len(afforestable_non_agri_non_other_ecosystem_classes)}")



In [ ]:
# Calculate area for each polygon (in m²) and then group by class
afforestable_non_agri_non_other_ecosystem_classes['area_m2'] = afforestable_non_agri_non_other_ecosystem_classes.geometry.area
area_by_class = afforestable_non_agri_non_other_ecosystem_classes.groupby('Classify')['area_m2'].sum()



## Merge afforestable non agri non other ecosystems classes with catchments 

In [ ]:
afforestable_non_agri_non_other_ecosystems_in_basins = gpd.overlay(afforestable_non_agri_non_other_ecosystem_classes, hydrobasins, how='intersection')

In [ ]:
# Define a color mapping for each class (adjust hex codes as desired)
color_dict = {
    'Fields: Bare Land': '#fdae61',
    'Quarry': '#d7191c',
    'Bauxite Extraction': '#2b83ba',
}


In [ ]:
# Set up the plot with common extents (using hydrobasins extent for context)
common_xlim = (593635.9271443005, 848114.2122774671)
common_ylim = (612896.835085367, 712816.0327676072)


In [ ]:
# Create the plot
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot hydrobasins as a background layer
hydrobasins.plot(ax=ax, color="white", edgecolor="blue")

# Plot each ecosystem class and build legend handles with area info
legend_handles = []
for cls, color in color_dict.items():
    subset = afforestable_non_agri_non_other_ecosystem_classes[
        afforestable_non_agri_non_other_ecosystem_classes["Classify"] == cls
    ]
    if not subset.empty:
        subset.plot(ax=ax, color=color, edgecolor="black", alpha=0.7, zorder=101)
        total_area = area_by_class.get(cls, 0)  # Total area in m²
        total_area_ha = total_area / 10000       # Convert to hectares
        label = f"{cls} ({total_area_ha:,.0f} ha)"
        legend_handles.append(mpatches.Patch(color=color, label=label))

# Optionally, add a legend entry for the hydrobasins background
bg_legend = Line2D([0], [0], color='blue', lw=2, label='Catchments')
legend_handles.insert(0, bg_legend)

# Define scale bar and north arrow functions
def add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, tick_height=0.01, label_offset=0.04, km_offset=0.01):
    x, y = location
    bar_half_length = 0.05
    ax.plot([x - bar_half_length, x + bar_half_length], [y, y],
            transform=ax.transAxes, color="black", linewidth=linewidth)
    for pos in [x - bar_half_length, x, x + bar_half_length]:
        ax.plot([pos, pos], [y - tick_height/2, y + tick_height/2],
                transform=ax.transAxes, color="black", linewidth=linewidth)
    ax.text(x - bar_half_length, y - tick_height - label_offset, "0",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x, y - tick_height - label_offset, f"{int(length_km // 2)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length, y - tick_height - label_offset, f"{int(length_km)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length + km_offset, y, "km",
            transform=ax.transAxes, ha="left", va="center", fontsize=12)

def add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03):
    x, y = location
    ax.annotate("", xy=(x, y + size), xycoords="axes fraction",
                xytext=(x, y), textcoords="axes fraction",
                arrowprops=dict(facecolor="black", edgecolor="black", headwidth=10, headlength=15, width=5))
    ax.text(x, y + size + label_offset, "N",
            transform=ax.transAxes, fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black")

# Add scale bar and north arrow to the map
add_scale_bar(ax, length_km=20, location=(0.9, 0.79))
add_north_arrow(ax, location=(0.9, 0.85))

# Set axis limits, labels, and title
ax.set_xlim(common_xlim)
ax.set_ylim(common_ylim)
ax.set_xlabel("Easting", fontsize=14, fontname="Times New Roman")
ax.set_ylabel("Northing", fontsize=14, fontname="Times New Roman")
ax.set_title("Afforestable non-agricultural lands", fontsize=20, fontweight="bold", fontname="Times New Roman", pad=20)

# Add the legend positioned beneath the plot
ax.legend(handles=legend_handles, title="Afforestable non-agricultural land classes", 
          loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=3, 
          frameon=False, fontsize=12, title_fontsize=14)

plt.tight_layout()
fig.savefig(map_forest_connectivity_figures_folder / "afforestable_non_agricultural_lands.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 3. Calculate geometry metrics for the forests (if not already available)
existing_and_future_non_agri_afforested_lands['area'] = existing_and_future_non_agri_afforested_lands.geometry.area
existing_and_future_non_agri_afforested_lands['perimeter'] = existing_and_future_non_agri_afforested_lands.geometry.length

In [ ]:
display(existing_and_future_non_agri_afforested_lands['area'] )

In [ ]:
# 4. Perform a spatial intersection to split forest patches by hydrobasin boundaries.
#    This ensures that if a forest patch crosses a hydrobasin boundary, the portions
#    are assigned to the respective HYBAS_ID.
existing_forest_and_future_non_agri_afforested_in_basins = gpd.overlay(existing_and_future_non_agri_afforested_lands, hydrobasins, how='intersection')

In [ ]:
# 5. Recalculate geometry metrics for the intersected polygons,
#    since the intersection may change their sizes and edge lengths.
existing_forest_and_future_non_agri_afforested_in_basins['area'] = existing_forest_and_future_non_agri_afforested_in_basins.geometry.area
existing_forest_and_future_non_agri_afforested_in_basins['perimeter'] = existing_forest_and_future_non_agri_afforested_in_basins.geometry.length

In [ ]:
# --- Compute aggregated forest statistics from the intersection ---
existing_and_non_agri_afforestable_forest_stats = existing_forest_and_future_non_agri_afforested_in_basins.groupby('HYBAS_ID').agg(
    patch_count=('area', 'count'),
    avg_patch_size=('area', 'mean'),
    largest_patch_size=('area', 'max'),
    total_edge_length=('perimeter', 'sum'),
    total_forest_area=('area', 'sum')
).reset_index()

# Convert forest areas from m² to hectares
existing_and_non_agri_afforestable_forest_stats['avg_patch_size_ha'] = existing_and_non_agri_afforestable_forest_stats['avg_patch_size'] / 10000
existing_and_non_agri_afforestable_forest_stats['largest_patch_size_ha'] = existing_and_non_agri_afforestable_forest_stats['largest_patch_size'] / 10000
existing_and_non_agri_afforestable_forest_stats['total_forest_area_ha'] = existing_and_non_agri_afforestable_forest_stats['total_forest_area'] / 10000

In [ ]:
# --- Merge aggregated stats with all hydrobasins ---
# Start with all hydrobasins (which has 103 rows) and merge forest_stats onto it.
existing_and_non_agri_afforestable_basin_stats = hydrobasins[['HYBAS_ID', 'geometry']].copy()
existing_and_non_agri_afforestable_basin_stats = existing_and_non_agri_afforestable_basin_stats.merge(existing_and_non_agri_afforestable_forest_stats, on='HYBAS_ID', how='left')  # left join ensures all catchments are kept

# Calculate the catchment area (in m²) for each hydrobasin
existing_and_non_agri_afforestable_basin_stats['catchment_area'] = existing_and_non_agri_afforestable_basin_stats.geometry.area

# --- Fill missing values for catchments with no forest ---
# These NaNs occur in catchments with no intersecting forest features.
existing_and_non_agri_afforestable_basin_stats['patch_count'] = existing_and_non_agri_afforestable_basin_stats['patch_count'].fillna(0)
existing_and_non_agri_afforestable_basin_stats['avg_patch_size'] = existing_and_non_agri_afforestable_basin_stats['avg_patch_size'].fillna(0)
existing_and_non_agri_afforestable_basin_stats['largest_patch_size'] = existing_and_non_agri_afforestable_basin_stats['largest_patch_size'].fillna(0)
existing_and_non_agri_afforestable_basin_stats['total_edge_length'] = existing_and_non_agri_afforestable_basin_stats['total_edge_length'].fillna(0)
existing_and_non_agri_afforestable_basin_stats['total_forest_area'] = existing_and_non_agri_afforestable_basin_stats['total_forest_area'].fillna(0)
existing_and_non_agri_afforestable_basin_stats['avg_patch_size_ha'] = existing_and_non_agri_afforestable_basin_stats['avg_patch_size_ha'].fillna(0)
existing_and_non_agri_afforestable_basin_stats['largest_patch_size_ha'] = existing_and_non_agri_afforestable_basin_stats['largest_patch_size_ha'].fillna(0)
existing_and_non_agri_afforestable_basin_stats['total_forest_area_ha'] = existing_and_non_agri_afforestable_basin_stats['total_forest_area_ha'].fillna(0)

In [ ]:
# --- Calculate forest percentage ---
# Make sure to use values in m² (both total_forest_area and catchment_area are in m²)
existing_and_non_agri_afforestable_basin_stats['pct_forest'] = (existing_and_non_agri_afforestable_basin_stats['total_forest_area'] / existing_and_non_agri_afforestable_basin_stats['catchment_area']) * 100
existing_and_non_agri_afforestable_basin_stats['pct_forest'] = existing_and_non_agri_afforestable_basin_stats['pct_forest'].fillna(0)  # set NaN to 0% for catchments with no forest

In [ ]:
# --- Add catchment area in hectares ---
existing_and_non_agri_afforestable_basin_stats['catchment_area_ha'] = existing_and_non_agri_afforestable_basin_stats['catchment_area'] / 10000

In [ ]:
# --- (Optional) Format numeric outputs with commas and two decimal places ---
# For example, you might do something like:
cols_to_format = ['patch_count', 'avg_patch_size', 'largest_patch_size', 'total_edge_length',
                  'avg_patch_size_ha', 'largest_patch_size_ha', 'total_forest_area',
                  'total_forest_area_ha', 'catchment_area', 'catchment_area_ha', 'pct_forest']
for col in cols_to_format:
    existing_and_non_agri_afforestable_basin_stats[col] = pd.to_numeric(existing_and_non_agri_afforestable_basin_stats[col], errors='coerce')  # CHANGED/ADDED

# Apply formatting to each column using no decimal places
existing_and_non_agri_afforestable_basin_stats['patch_count'] = existing_and_non_agri_afforestable_basin_stats['patch_count'].apply(lambda x: "{:,.0f}".format(x))
existing_and_non_agri_afforestable_basin_stats['avg_patch_size'] = existing_and_non_agri_afforestable_basin_stats['avg_patch_size'].apply(lambda x: "{:,.0f}".format(x))               # CHANGED: whole numbers
existing_and_non_agri_afforestable_basin_stats['largest_patch_size'] = existing_and_non_agri_afforestable_basin_stats['largest_patch_size'].apply(lambda x: "{:,.0f}".format(x))           # CHANGED: whole numbers
existing_and_non_agri_afforestable_basin_stats['total_edge_length'] = existing_and_non_agri_afforestable_basin_stats['total_edge_length'].apply(lambda x: "{:,.0f}".format(x))             # CHANGED: whole numbers
existing_and_non_agri_afforestable_basin_stats['avg_patch_size_ha'] = existing_and_non_agri_afforestable_basin_stats['avg_patch_size_ha'].apply(lambda x: "{:,.0f}".format(x))             # CHANGED: whole numbers
existing_and_non_agri_afforestable_basin_stats['largest_patch_size_ha'] = existing_and_non_agri_afforestable_basin_stats['largest_patch_size_ha'].apply(lambda x: "{:,.0f}".format(x))         # CHANGED: whole numbers
existing_and_non_agri_afforestable_basin_stats['total_forest_area'] = existing_and_non_agri_afforestable_basin_stats['total_forest_area'].apply(lambda x: "{:,.0f}".format(x))             # CHANGED: whole numbers
existing_and_non_agri_afforestable_basin_stats['total_forest_area_ha'] = existing_and_non_agri_afforestable_basin_stats['total_forest_area_ha'].apply(lambda x: "{:,.0f}".format(x))           # CHANGED: whole numbers
existing_and_non_agri_afforestable_basin_stats['catchment_area'] = existing_and_non_agri_afforestable_basin_stats['catchment_area'].apply(lambda x: "{:,.0f}".format(x))                     # CHANGED: whole numbers
existing_and_non_agri_afforestable_basin_stats['catchment_area_ha'] = existing_and_non_agri_afforestable_basin_stats['catchment_area_ha'].apply(lambda x: "{:,.0f}".format(x))               # CHANGED: whole numbers
existing_and_non_agri_afforestable_basin_stats['pct_forest'] = existing_and_non_agri_afforestable_basin_stats['pct_forest'].apply(lambda x: "{:,.0f}".format(x))                             # CHANGED: whole numbers

# Rename the columns as requested
existing_and_non_agri_afforestable_basin_stats = existing_and_non_agri_afforestable_basin_stats.rename(columns={
    "patch_count": "Number of forest patches",
    "avg_patch_size": "Average patch size (m2)",
    "largest_patch_size": "Largest patch size (m2)",
    "total_edge_length": "Total edge length",
    "avg_patch_size_ha": "Average patch size (ha)",
    "largest_patch_size_ha": "Largest patch size (ha)",
    "total_forest_area": "Total catchment forest area (m2)",  # CHANGED: Renamed to reflect catchment forest area
    "total_forest_area_ha": "Total catchment forest area (ha)",   # CHANGED: Renamed accordingly
    "catchment_area": "Catchment area (m2)",
    "catchment_area_ha": "Catchment area (ha)",                  # ADDED: New column for hectares
    "pct_forest": "Percentage forest in catchment"             # CHANGED: Renamed 'pct_forest' to the final column name
})


In [ ]:
# Optionally, inspect or export your results
display(existing_and_non_agri_afforestable_basin_stats)

In [ ]:
existing_and_non_agri_afforestable_basin_stats.to_csv(output_folder / "existing_and_non_agri_afforestable_basin_stats.csv", index=False)

In [ ]:
# Compute NN statistics on the individual forest patches in each catchment
def compute_nn_stats(group):
    # Compute centroids of the forest patches in this catchment
    centroids = group.geometry.centroid
    # If there are fewer than 2 patches, NN distances cannot be computed
    if len(centroids) < 2:
        return pd.Series({
            'min_nn_distance': np.nan,
            'mean_nn_distance': np.nan,
            'max_nn_distance': np.nan
        })
    # Create an array of (x, y) coordinates from the centroids
    coords = np.array([(pt.x, pt.y) for pt in centroids])
    
    # Build a KDTree for fast nearest neighbor search
    tree = cKDTree(coords)
    # Query for each point: k=2 returns the point itself and its nearest neighbor
    distances, indices = tree.query(coords, k=2)
    nn_distances = distances[:, 1]  # take the second column (nearest neighbor distance)
    
    # CHANGED: Round the NN distances to whole numbers and format with commas
    return pd.Series({
        'min_nn_distance': "{:,.0f}".format(nn_distances.min()),
        'mean_nn_distance': "{:,.0f}".format(nn_distances.mean()),
        'max_nn_distance': "{:,.0f}".format(nn_distances.max())
    })

# First, apply the function to compute nn_stats
nn_stats = existing_forest_and_future_non_agri_afforested_in_basins.groupby("HYBAS_ID").apply(compute_nn_stats).reset_index()

# Now, rename the columns as requested
nn_stats = nn_stats.rename(columns={
    "min_nn_distance": "Minimum nearest neighbour distance (m)",
    "mean_nn_distance": "Mean nearest neighbour distance (m)",
    "max_nn_distance": "Maximum nearest neighbour distance (m)",
})

# Inspect the resulting NN statistics
display(nn_stats.head())

In [ ]:
# Merge the NN statistics with the existing basin stats on HYBAS_ID
final_stats = existing_and_non_agri_afforestable_basin_stats.merge(nn_stats, on="HYBAS_ID", how="left")

# CHANGED/ADDED: Drop columns with "(m2)" in the column name, keeping only the (ha) columns (and others)
cols_to_drop = [col for col in final_stats.columns if "(m2)" in col]
final_stats_reduced = final_stats.drop(columns=cols_to_drop)

# Optionally, inspect the resulting columns
display("Final columns to be exported:")
display(final_stats_reduced.columns)

# Save the merged DataFrame as a CSV in your output folder
output_csv_path = output_folder / "final_basin_stats.csv"
final_stats.to_csv(output_csv_path, index=False)

# Optionally, inspect the final results
display(final_stats_reduced.head())

In [ ]:
# Merge the NN statistics with the existing basin stats on HYBAS_ID
final_stats = existing_and_non_agri_afforestable_basin_stats.merge(nn_stats, on="HYBAS_ID", how="left")

# CHANGED/ADDED: Drop columns with "(m2)" in the column name, keeping only the (ha) columns (and others)
cols_to_drop = [col for col in final_stats.columns if "(m2)" in col]
final_stats_reduced = final_stats.drop(columns=cols_to_drop)

# Optionally, inspect the resulting columns
display("Final columns to be exported:")
display(final_stats_reduced.columns)

# Save the merged DataFrame as a CSV in your output folder
output_csv_path = output_folder / "final_basin_stats.csv"
final_stats.to_csv(output_csv_path, index=False)

# Optionally, inspect the final results
display(final_stats_reduced.head())

In [ ]:
display("Summary of 'Number of forest patches':")
display(final_stats["Number of forest patches"].describe())

# Ensure that the "Percentage forest in catchment" column is numeric.
# If it's a formatted string with commas, remove them and convert to numeric.
if final_stats["Number of forest patches"].dtype == "object":
    final_stats["Number of forest patches"] = pd.to_numeric(
        final_stats["Number of forest patches"].str.replace(",", ""),
        errors='coerce'
    )

# Also, ensure HYBAS_ID is treated as a categorical variable or string
final_stats["HYBAS_ID"] = final_stats["HYBAS_ID"].astype(str)

# Create a bar chart of "Percentage forest in catchment" vs. HYBAS_ID
plt.figure(figsize=(12,6))
plt.bar(final_stats["HYBAS_ID"], final_stats["Number of forest patches"], color='skyblue')
plt.xlabel("HYBAS_ID")
plt.ylabel("Number of forest patches")
plt.title("Number of forest patches")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
display("Summary of 'Percentage forest in catchment':")
display(final_stats["Percentage forest in catchment"].describe())

# Ensure that the "Percentage forest in catchment" column is numeric.
# If it's a formatted string with commas, remove them and convert to numeric.
if final_stats["Percentage forest in catchment"].dtype == "object":
    final_stats["Percentage forest in catchment"] = pd.to_numeric(
        final_stats["Percentage forest in catchment"].str.replace(",", ""),
        errors='coerce'
    )

# Also, ensure HYBAS_ID is treated as a categorical variable or string
final_stats["HYBAS_ID"] = final_stats["HYBAS_ID"].astype(str)

# Create a bar chart of "Percentage forest in catchment" vs. HYBAS_ID
plt.figure(figsize=(12,6))
plt.bar(final_stats["HYBAS_ID"], final_stats["Percentage forest in catchment"], color='skyblue')
plt.xlabel("HYBAS_ID")
plt.ylabel("Percentage forest in catchment (%)")
plt.title("Percentage Forest in Catchment by HYBAS_ID")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
display("Summary of 'Average patch size (ha)':")
display(final_stats["Average patch size (ha)"].describe())

# Ensure that the "Percentage forest in catchment" column is numeric.
# If it's a formatted string with commas, remove them and convert to numeric.
if final_stats["Average patch size (ha)"].dtype == "object":
    final_stats["Average patch size (ha)"] = pd.to_numeric(
        final_stats["Average patch size (ha)"].str.replace(",", ""),
        errors='coerce'
    )

# Also, ensure HYBAS_ID is treated as a categorical variable or string
final_stats["HYBAS_ID"] = final_stats["HYBAS_ID"].astype(str)

# Create a bar chart of "Percentage forest in catchment" vs. HYBAS_ID
plt.figure(figsize=(12,6))
plt.bar(final_stats["HYBAS_ID"], final_stats["Average patch size (ha)"], color='skyblue')
plt.xlabel("HYBAS_ID")
plt.ylabel("Average patch size (ha)")
plt.title("Average patch size (ha) by HYBAS_ID")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Debug: Check that final_stats has data and the expected columns
display("Summary of 'Maximum nearest neighbour distance (m)':")
display(final_stats["Maximum nearest neighbour distance (m)"].describe())

# Ensure that the "Percentage forest in catchment" column is numeric.
# If it's a formatted string with commas, remove them and convert to numeric.
if final_stats["Maximum nearest neighbour distance (m)"].dtype == "object":
    final_stats["Maximum nearest neighbour distance (m)"] = pd.to_numeric(
        final_stats["Maximum nearest neighbour distance (m)"].str.replace(",", ""),
        errors='coerce'
    )

# Also, ensure HYBAS_ID is treated as a categorical variable or string
final_stats["HYBAS_ID"] = final_stats["HYBAS_ID"].astype(str)

# Create a bar chart of "Percentage forest in catchment" vs. HYBAS_ID
plt.figure(figsize=(12,6))
plt.bar(final_stats["HYBAS_ID"], final_stats["Maximum nearest neighbour distance (m)"], color='skyblue')
plt.xlabel("HYBAS_ID")
plt.ylabel("Maximum nearest neighbour distance (m)")
plt.title("Maximum nearest neighbour distance (m) by HYBAS_ID")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
display("Summary of 'Mean nearest neighbour distance (m)':")
display(final_stats["Mean nearest neighbour distance (m)"].describe())

# Ensure that the "Percentage forest in catchment" column is numeric.
# If it's a formatted string with commas, remove them and convert to numeric.
if final_stats["Mean nearest neighbour distance (m)"].dtype == "object":
    final_stats["Mean nearest neighbour distance (m)"] = pd.to_numeric(
        final_stats["Mean nearest neighbour distance (m)"].str.replace(",", ""),
        errors='coerce'
    )

# Also, ensure HYBAS_ID is treated as a categorical variable or string
final_stats["HYBAS_ID"] = final_stats["HYBAS_ID"].astype(str)

# Create a bar chart of "Percentage forest in catchment" vs. HYBAS_ID
plt.figure(figsize=(12,6))
plt.bar(final_stats["HYBAS_ID"], final_stats["Mean nearest neighbour distance (m)"], color='skyblue')
plt.xlabel("HYBAS_ID")
plt.ylabel("Mean nearest neighbour distance (m)")
plt.title("Mean nearest neighbour distance (m) by HYBAS_ID")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
display("Summary of 'Total edge length':")
display(final_stats["Total edge length"].describe())

# Ensure that the "Percentage forest in catchment" column is numeric.
# If it's a formatted string with commas, remove them and convert to numeric.
if final_stats["Total edge length"].dtype == "object":
    final_stats["Total edge length"] = pd.to_numeric(
        final_stats["Total edge length"].str.replace(",", ""),
        errors='coerce'
    )

# Also, ensure HYBAS_ID is treated as a categorical variable or string
final_stats["HYBAS_ID"] = final_stats["HYBAS_ID"].astype(str)

# Create a bar chart of "Percentage forest in catchment" vs. HYBAS_ID
plt.figure(figsize=(12,6))
plt.bar(final_stats["HYBAS_ID"], final_stats["Total edge length"], color='skyblue')
plt.xlabel("HYBAS_ID")
plt.ylabel("Total edge length")
plt.title("Total edge length")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()